# NLP Lab Experiments 6–9

In [ ]:
!pip -q install nltk spacy stanza scikit-learn pandas
!python -m spacy download en_core_web_sm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.2/794.2 kB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 35.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 102.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
!pip -q install stanza

## EXPERIMENT 6 — N-gram Next-Word Prediction
Input: Corpus: I love natural language processing. I love machine learning. Natural language processing is interesting.



In [ ]:
# EXPERIMENT 6A — NLTK implementation
import re
import math
from collections import Counter, defaultdict
import nltk
from nltk.tokenize import word_tokenize

nltk.download("punkt", quiet=True)
try:
    nltk.download("punkt_tab", quiet=True)
except:
    pass

corpus = input("Enter corpus: ")
sentence = input("Enter sentence for next-word prediction: ")

tokens = [t.lower() for t in word_tokenize(corpus) if re.search(r"[A-Za-z0-9]", t)]
query = [t.lower() for t in word_tokenize(sentence) if re.search(r"[A-Za-z0-9]", t)]

def build_ngram(tokens, n):
    counts = Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))
    context_counts = Counter(tuple(tokens[i:i+n-1]) for i in range(len(tokens)-n+1))
    return counts, context_counts

def predict(tokens, query, n, alpha=1.0):
    ngram_counts, context_counts = build_ngram(tokens, n)
    context = tuple(query[-(n-1):]) if n > 1 else tuple()
    vocab = set(tokens)
    candidates = []
    for ng, c in ngram_counts.items():
        if ng[:-1] == context:
            # Laplace smoothing
            prob = (c + alpha) / (context_counts[context] + alpha * len(vocab))
            candidates.append((ng[-1], prob, c))
    if not candidates:
        return None, []
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[0][0], candidates

for n in [2, 3, 4]:
    pred, candidates = predict(tokens, query, n)
    print(f"\n{n}-gram:")
    if pred:
        print("Predicted next word:", pred)
        print("Probability:", round(candidates[0][1], 4))
        print("Candidate probabilities:")
        for word, p, c in candidates[:10]:
            print(f"  {word:15} count={c:3} probability={p:.4f}")
    else:
        print("No matching context found in the corpus.")

Enter corpus: I love natural language processing. I love machine learning. Natural language processing is interesting.
Enter sentence for next-word prediction: I love

2-gram:
Predicted next word: natural
Probability: 0.1818
Candidate probabilities:
  natural         count=  1 probability=0.1818
  machine         count=  1 probability=0.1818

3-gram:
Predicted next word: natural
Probability: 0.1818
Candidate probabilities:
  natural         count=  1 probability=0.1818
  machine         count=  1 probability=0.1818

4-gram:
No matching context found in the corpus.


In [ ]:
# EXPERIMENT 6B — spaCy implementation
import spacy
from collections import Counter

nlp = spacy.load("en_core_web_sm")

corpus = input("Enter corpus: ")
sentence = input("Enter sentence for next-word prediction: ")

doc = nlp(corpus)
tokens = [t.text.lower() for t in doc if not t.is_space and t.is_alpha]
qdoc = nlp(sentence)
query = [t.text.lower() for t in qdoc if not t.is_space and t.is_alpha]

def spacy_ngram_predict(tokens, query, n, alpha=1.0):
    ng = Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))
    ctx = Counter(tuple(tokens[i:i+n-1]) for i in range(len(tokens)-n+1))
    vocab = set(tokens)
    context = tuple(query[-(n-1):])
    candidates = []
    for gram, count in ng.items():
        if gram[:-1] == context:
            p = (count + alpha) / (ctx[context] + alpha * len(vocab))
            candidates.append((gram[-1], p))
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates

for n in (2, 3, 4):
    candidates = spacy_ngram_predict(tokens, query, n)
    print(f"\n{n}-gram:")
    if candidates:
        print("Predicted next word:", candidates[0][0])
        print("Probability:", round(candidates[0][1], 4))
        for word, p in candidates[:10]:
            print(f"  {word:15} probability={p:.4f}")
    else:
        print("No matching context found.")

Enter corpus: I love natural language processing. I love machine learning. Natural language processing is interesting.
Enter sentence for next-word prediction: I love

2-gram:
Predicted next word: natural
Probability: 0.1818
  natural         probability=0.1818
  machine         probability=0.1818

3-gram:
Predicted next word: natural
Probability: 0.1818
  natural         probability=0.1818
  machine         probability=0.1818

4-gram:
No matching context found.


### Stanford NLP option


In [ ]:
# EXPERIMENT 6C — Stanza N-gram
import stanza
from collections import Counter

stanza.download("en", verbose=False)

nlp = stanza.Pipeline(
    "en",
    processors="tokenize",
    use_gpu=False,
    verbose=False
)

corpus = input("Enter corpus: ")
sentence = input("Enter sentence for next-word prediction: ")

def tokenize(text):
    doc = nlp(text)
    return [
        word.text.lower()
        for sent in doc.sentences
        for word in sent.words
        if word.text.isalnum()
    ]

tokens = tokenize(corpus)
query = tokenize(sentence)

def predict_ngram(tokens, query, n):
    ngrams = Counter(
        tuple(tokens[i:i+n])
        for i in range(len(tokens)-n+1)
    )

    contexts = Counter(
        tuple(tokens[i:i+n-1])
        for i in range(len(tokens)-n+1)
    )

    context = tuple(query[-(n-1):])

    candidates = []

    for gram, count in ngrams.items():
        if gram[:-1] == context:
            probability = count / contexts[context]
            candidates.append(
                (gram[-1], probability)
            )

    candidates.sort(key=lambda x: x[1], reverse=True)

    return candidates

for n in [2, 3, 4]:

    results = predict_ngram(tokens, query, n)

    print(f"\n{n}-gram:")

    if results:
        print("Predicted word:", results[0][0])
        print("Probability:", round(results[0][1], 4))

        print("\nAll candidates:")
        for word, probability in results:
            print(
                f"{word:15} "
                f"{probability:.4f}"
            )
    else:
        print("No matching context found.")

Enter corpus: I love natural language processing. I love machine learning. Natural language processing is interesting.
Enter sentence for next-word prediction: I love

2-gram:
Predicted word: natural
Probability: 0.5

All candidates:
natural         0.5000
machine         0.5000

3-gram:
Predicted word: natural
Probability: 0.5

All candidates:
natural         0.5000
machine         0.5000

4-gram:
No matching context found.


## EXPERIMENT 7 — POS Tagging
input: I saw the man with the telescope.
Visiting relatives can be boring.
The old man the boats.
मैंने दूरबीन से आदमी को देखा।
रामने श्यामला पुस्तकासोबत पाहिले.

In [ ]:
# EXPERIMENT 7A — NLTK POS tagging
import nltk
from nltk.tokenize import word_tokenize

for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except:
        pass

sentence = input("Enter an ambiguous sentence: ")
tokens = word_tokenize(sentence)
tags = nltk.pos_tag(tokens)

print("\nNLTK POS Tags:")
for word, tag in tags:
    print(f"{word:15} -> {tag}")

Enter an ambiguous sentence: I saw the man with the telescope. Visiting relatives can be boring. The old man the boats. 

NLTK POS Tags:
I               -> PRP
saw             -> VBD
the             -> DT
man             -> NN
with            -> IN
the             -> DT
telescope       -> NN
.               -> .
Visiting        -> VBG
relatives       -> NNS
can             -> MD
be              -> VB
boring          -> VBG
.               -> .
The             -> DT
old             -> JJ
man             -> NN
the             -> DT
boats           -> NNS
.               -> .


In [ ]:
# EXPERIMENT 7B — spaCy POS tagging
import spacy
nlp = spacy.load("en_core_web_sm")

sentence = input("Enter an ambiguous sentence: ")
doc = nlp(sentence)

print("\nspaCy POS Tags:")
for token in doc:
    print(f"{token.text:15} -> {token.pos_:8} ({token.tag_})")

Enter an ambiguous sentence: I saw the man with the telescope. Visiting relatives can be boring. The old man the boats. 

spaCy POS Tags:
I               -> PRON     (PRP)
saw             -> VERB     (VBD)
the             -> DET      (DT)
man             -> NOUN     (NN)
with            -> ADP      (IN)
the             -> DET      (DT)
telescope       -> NOUN     (NN)
.               -> PUNCT    (.)
Visiting        -> VERB     (VBG)
relatives       -> NOUN     (NNS)
can             -> AUX      (MD)
be              -> AUX      (VB)
boring          -> ADJ      (JJ)
.               -> PUNCT    (.)
The             -> DET      (DT)
old             -> ADJ      (JJ)
man             -> NOUN     (NN)
the             -> DET      (DT)
boats           -> NOUN     (NNS)
.               -> PUNCT    (.)


In [ ]:
# EXPERIMENT 7C — Stanza POS Tagging
import stanza

# Download English model
stanza.download("en")

# Create pipeline
nlp = stanza.Pipeline(
    "en",
    processors="tokenize,pos",
    use_gpu=False
)

sentence = input("Enter an ambiguous sentence: ")

doc = nlp(sentence)

print("\nStanza POS Tags:")
for sent in doc.sentences:
    for word in sent.words:
        print(f"{word.text:15} -> {word.upos:10} ({word.xpos})")

INFO:stanza:Downloaded file to /root/.cache/stanza/1.14.0/resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...
INFO:stanza:File exists: /root/.cache/stanza/1.14.0/resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.14.0/resources
INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/.cache/stanza/1.14.0/resources/resources.json
INFO:stanza:Loading these models for language: en (English):
| Processor | Package           |
---------------------------------
| tokenize  | combined_nocharlm |
| mwt       | combined          |
| pos       | combined_charlm   |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: mwt
INFO:stanza:Loading: pos
INFO:stanza:Done loading processors!


Enter an ambiguous sentence: I saw the man with the telescope. Visiting relatives can be boring. The old man the boats. 

Stanza POS Tags:
I               -> PRON       (PRP)
saw             -> VERB       (VBD)
the             -> DET        (DT)
man             -> NOUN       (NN)
with            -> ADP        (IN)
the             -> DET        (DT)
telescope       -> NOUN       (NN)
.               -> PUNCT      (.)
Visiting        -> VERB       (VBG)
relatives       -> NOUN       (NNS)
can             -> AUX        (MD)
be              -> AUX        (VB)
boring          -> ADJ        (JJ)
.               -> PUNCT      (.)
The             -> DET        (DT)
old             -> ADJ        (JJ)
man             -> NOUN       (NN)
the             -> DET        (DT)
boats           -> NOUN       (NNS)
.               -> PUNCT      (.)


In [ ]:
# EXPERIMENT 7D — Regional-language POS tagging with Stanza
# Stanza is the current Stanford NLP group ecosystem for multilingual NLP.
import stanza

lang = input("Enter language code (hi=Hindi, mr=Marathi etc.): ").strip().lower()
text = input("Enter a regional-language sentence: ")

try:
    stanza.download(lang, processors="tokenize,pos", verbose=False)
    pipe = stanza.Pipeline(lang=lang, processors="tokenize,pos", use_gpu=False, verbose=False)
    doc = pipe(text)
    print("\nRegional-language POS Tags:")
    for sent in doc.sentences:
        for w in sent.words:
            print(f"{w.text:15} -> {w.upos:8} ({w.xpos})")
except Exception as e:
    print("Could not load this language/model:", e)

Enter language code (hi=Hindi, mr=Marathi etc.): mr
Enter a regional-language sentence: रामने श्यामला पुस्तकासोबत पाहिले.

Regional-language POS Tags:
रामने           -> NOUN     (None)
श्यामला         -> NOUN     (None)
पुस्तकासोबत     -> NOUN     (None)
पाहिले          -> VERB     (None)
.               -> PUNCT    (None)


## EXPERIMENT 9 — Named Entity Recognition (NER)
input: John working at IBM wants to buy a new Samsung phone from Bangalore.

राहुल ने मुंबई में टाटा कंसल्टेंसी सर्विसेज में नौकरी शुरू की।
अमितने पुण्यात टाटा मोटर्समध्ये नोकरी सुरू केली.

In [ ]:
    # EXPERIMENT 9A — NLTK NER
    import nltk
    from nltk import word_tokenize, pos_tag, ne_chunk

    for pkg in ["maxent_ne_chunker", "words", "maxent_ne_chunker_tab"]:
        nltk.download(pkg, quiet=True)

    sentence = input("Enter sentence: ")
    tokens = word_tokenize(sentence)
    tags = pos_tag(tokens)
    tree = ne_chunk(tags)

    print("\nNLTK Named Entities:")
    for subtree in tree:
        if hasattr(subtree, "label"):
            entity = " ".join(word for word, tag in subtree.leaves())
            print(f"{entity:25} -> {subtree.label()}")

Enter sentence: Ram bought a Samsung phone at 10% discount at 40000 Rs while visiting Delhi for CommonWealth Games. He also went for the BRICS event.

NLTK Named Entities:
Ram                       -> PERSON
Samsung                   -> ORGANIZATION
Delhi                     -> PERSON
CommonWealth Games        -> ORGANIZATION
BRICS                     -> ORGANIZATION


In [ ]:
# EXPERIMENT 9B — spaCy NER
import spacy
nlp = spacy.load("en_core_web_sm")

sentence = input("Enter sentence: ")
doc = nlp(sentence)

print("\nspaCy Named Entities:")
for ent in doc.ents:
    print(f"{ent.text:25} -> {ent.label_}")

Enter sentence:  Ram bought a Samsung phone at 10% discount at 40000 Rs while visiting Delhi for CommonWealth Games. He also went for the BRICS event.

spaCy Named Entities:
Ram                       -> ORG
Samsung                   -> ORG
10%                       -> PERCENT
40000                     -> CARDINAL
Delhi                     -> GPE
CommonWealth Games        -> ORG
BRICS                     -> GPE


In [ ]:
# EXPERIMENT 9C — Stanza NER

!pip -q install stanza
import stanza

stanza.download("en", verbose=False)

nlp = stanza.Pipeline(
    "en",
    processors="tokenize,ner",
    use_gpu=False,
    verbose=False
)

sentence = input("Enter sentence: ")

doc = nlp(sentence)

print("\nStanza Named Entities:")

for ent in doc.ents:

    print(
        f"{ent.text:25} -> {ent.type}"
    )

Enter sentence: Ram bought a Samsung phone at 10% discount at 40000 Rs while visiting Delhi for CommonWealth Games. He also went for the BRICS 2026 Summit.

Stanza Named Entities:
Ram                       -> PERSON
Samsung                   -> ORG
10%                       -> PERCENT
40000 Rs                  -> MONEY
Delhi                     -> GPE
CommonWealth Games        -> EVENT
the BRICS 2026 Summit     -> EVENT


In [ ]:
# EXPERIMENT 9D — Regional-language NER using Stanza
lang = input("Language code (hi/mr/): ").strip().lower()
text = input("Enter regional-language sentence: ")

try:
    stanza.download(lang, processors="tokenize,ner", verbose=False)
    pipe = stanza.Pipeline(lang=lang, processors="tokenize,ner", use_gpu=False, verbose=False)
    doc = pipe(text)

    print("\nRegional-language NER:")
    for ent in doc.ents:
        print(f"{ent.text:25} -> {ent.type}")
except Exception as e:
    print("NER model unavailable for this language:", e)

Language code (hi/mr/): mr
Enter regional-language sentence: अमितने पुण्यात टाटा मोटर्समध्ये नोकरी सुरू केली.

Regional-language NER:
अमितने                    -> NEP
पुण्यात                   -> NEL
टाटा मोटर्समध्ये          -> NEO


inputs

**Experiment 6:**  
`Corpus: I love natural language processing. I love machine learning. Natural language processing is interesting.`  


**Experiment 7:**  
`I saw the man with the telescope.`  
`Visiting relatives can be boring.`  
`The old man the boats.`

**Experiment 8:**  
`The small brown dog chased the red ball in the garden.`

**Experiment 9:**  
`John working at IBM wants to buy a new Samsung phone from Bangalore.`

For Experiment 9: John–Person, IBM–Organisation, Samsung–Product name and Bangalore–Location.
